# Consecutive-groups viewer — v62 vs v63

Browse the `datasets/consecutive_groups` predictions **frame by frame** for a
consecutive group of 11 images, comparing the two fine-tuned checkpoints:

- **v62** — `workspace/consecutive_v62_inference/epoch_009`
- **v63** — `workspace/consecutive_v63_inference/epoch_004` (still running in another window)

The main cell renders an interactive Plotly figure with three panels:
**RGB | v62 depth | v63 depth**. Hover any pixel to read `(x, y, depth_m)` — the
same hover-to-measure behaviour as `kbz_inference_viewer.ipynb`.

Use the **group** dropdown and the **frame** slider (0–10) to scrub through a
consecutive group like a flip-book. Because v63 is still writing outputs, the
group list is intersected with what's finished on disk, so only groups where
*both* runs have all 11 frames appear. Re-run the discovery cell to pick up
newly-finished groups.

Works over Remote-SSH / port-forwarded Jupyter. Requires `ipywidgets` + `plotly`.

In [1]:
import re
from pathlib import Path

import numpy as np
from PIL import Image as PILImage

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

# --- What we're comparing ---------------------------------------------------
RGB_DIR = REPO_ROOT / 'datasets' / 'consecutive_groups'
V62_DIR = REPO_ROOT / 'workspace' / 'consecutive_v62_inference' / 'epoch_009'
V63_DIR = REPO_ROOT / 'workspace' / 'consecutive_v63_inference' / 'epoch_004'

# Predicted depth is treated as raw metric meters (this dataset carries no
# focal metadata and values already sit in a sensible ~0.25-7 m range). If a
# focal rescale turns out to be needed, set this to 7200/300 = 24 etc.
DEPTH_SCALE = 1.0

GROUP_SIZE = 11  # frames per consecutive group


def load_pred(run_dir: Path, stem: str) -> np.ndarray:
    """Load a prediction .npy for a stem from a run dir, in real-world meters."""
    return np.load(run_dir / 'depth_npy' / f'{stem}.npy').astype(np.float32) * DEPTH_SCALE


def rgb_path(stem: str) -> Path | None:
    for ext in ('.jpg', '.jpeg', '.png'):
        p = RGB_DIR / f'{stem}{ext}'
        if p.is_file():
            return p
    return None


print('RGB_DIR :', RGB_DIR)
print('V62_DIR :', V62_DIR)
print('V63_DIR :', V63_DIR)
for d in (RGB_DIR, V62_DIR, V63_DIR):
    assert d.is_dir(), f'not found: {d}'

RGB_DIR : /home/derek_austin/Depth-Anything-3/datasets/consecutive_groups
V62_DIR : /home/derek_austin/Depth-Anything-3/workspace/consecutive_v62_inference/epoch_009
V63_DIR : /home/derek_austin/Depth-Anything-3/workspace/consecutive_v63_inference/epoch_004


## Discover consecutive groups

Group stems by their `gNNN` prefix and keep only groups where **both** v62 and
v63 have written all 11 depth `.npy` files. Frames within a group are sorted by
their trailing timestamp so the slider scrubs in temporal order.

Re-run this cell as v63 finishes more groups to expand the list.

In [2]:
GROUP_RE = re.compile(r'^(g\d+)_')


def stems_in(run_dir: Path) -> set[str]:
    return {p.stem for p in (run_dir / 'depth_npy').glob('*.npy')}


v62_stems = stems_in(V62_DIR)
v63_stems = stems_in(V63_DIR)
common = v62_stems & v63_stems  # only stems both runs have predicted

# stem -> group id; group id -> sorted list of stems
groups: dict[str, list[str]] = {}
for stem in common:
    m = GROUP_RE.match(stem)
    if m:
        groups.setdefault(m.group(1), []).append(stem)


def frame_sort_key(stem: str):
    # stem looks like g001_000314_0001782081664978262 -> sort by trailing ts
    parts = stem.split('_')
    return int(parts[-1]) if parts[-1].isdigit() else stem


for gid in groups:
    groups[gid].sort(key=frame_sort_key)

# Keep only fully-complete groups (all GROUP_SIZE frames present in both runs).
complete = {gid: frames for gid, frames in groups.items() if len(frames) == GROUP_SIZE}
GROUP_IDS = sorted(complete)

print(f'v62 stems           : {len(v62_stems)}')
print(f'v63 stems           : {len(v63_stems)}  (still running)')
print(f'common stems        : {len(common)}')
print(f'complete groups (11): {len(GROUP_IDS)} / {len(groups)}')
print(f'first / last group  : {GROUP_IDS[0]} .. {GROUP_IDS[-1]}')
assert GROUP_IDS, 'no fully-complete groups found in both runs yet'

v62 stems           : 2002
v63 stems           : 2002  (still running)
common stems        : 2002
complete groups (11): 182 / 182
first / last group  : g001 .. g182


## Frame-by-frame viewer: RGB | v62 | v63

- **group** dropdown — pick a consecutive group (`gNNN`).
- **frame** slider — scrub `0 .. 10` through that group like a flip-book.

Three panels share pixel coordinates. Hover the RGB (via an invisible depth
overlay) or either depth map to read `depth = … m` at that pixel. The two depth
panels share one colour scale (2/98 percentile of the *union* of both preds for
that frame) so v62 and v63 are directly comparable by colour, not just by hover.

The title reports per-frame medians and the v63 − v62 median delta.

In [3]:
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

# Invisible colorscale: put a transparent heatmap over the RGB so hover works
# there too (go.Image carries no interpolated depth values).
INVIS = [[0, 'rgba(0,0,0,0)'], [1, 'rgba(0,0,0,0)']]

group_dd = widgets.Dropdown(options=GROUP_IDS, value=GROUP_IDS[0], description='group')
frame_sl = widgets.IntSlider(min=0, max=GROUP_SIZE - 1, step=1, value=0,
                             description='frame', continuous_update=False)
play = widgets.Play(min=0, max=GROUP_SIZE - 1, step=1, interval=600,
                    description='play')
widgets.jslink((play, 'value'), (frame_sl, 'value'))

fig = go.FigureWidget(
    make_subplots(
        rows=1, cols=3, horizontal_spacing=0.04,
        subplot_titles=('RGB', 'v62 depth', 'v63 depth'),
    )
)
# Trace order: 0 RGB image, 1 invisible-hover heatmap, 2 v62 depth, 3 v63 depth.
fig.add_trace(go.Image(z=np.zeros((2, 2, 3), dtype=np.uint8), hoverinfo='skip'), row=1, col=1)
fig.add_trace(go.Heatmap(z=[[0]], colorscale=INVIS, showscale=False,
                         hovertemplate='x=%{x} y=%{y}<br>depth=%{z:.2f} m<extra>RGB</extra>'),
              row=1, col=1)
fig.add_trace(go.Heatmap(z=[[0]], colorscale='Inferno', showscale=False,
                         hovertemplate='x=%{x} y=%{y}<br>depth=%{z:.2f} m<extra>v62</extra>'),
              row=1, col=2)
fig.add_trace(go.Heatmap(z=[[0]], colorscale='Inferno',
                         colorbar=dict(title='depth (m)', x=1.005),
                         hovertemplate='x=%{x} y=%{y}<br>depth=%{z:.2f} m<extra>v63</extra>'),
              row=1, col=3)
fig.update_layout(height=440, width=1320, margin=dict(t=70, l=30, r=60, b=30))


def render(gid: str, frame_idx: int):
    stem = complete[gid][frame_idx]
    v62 = load_pred(V62_DIR, stem)
    v63 = load_pred(V63_DIR, stem)
    H, W = v62.shape

    p = rgb_path(stem)
    rgb = (np.array(PILImage.open(p).convert('RGB').resize((W, H)))
           if p is not None else np.zeros((H, W, 3), dtype=np.uint8))

    # Shared colour scale from the union of both preds (fair comparison).
    both = np.concatenate([v62.ravel(), v63.ravel()])
    lo, hi = float(np.percentile(both, 2)), float(np.percentile(both, 98))

    with fig.batch_update():
        fig.data[0].z = rgb
        fig.data[1].z = v62  # invisible overlay: report v62 depth over RGB
        fig.data[2].z = v62
        fig.data[2].zmin, fig.data[2].zmax = lo, hi
        fig.data[3].z = v63
        fig.data[3].zmin, fig.data[3].zmax = lo, hi
        for col in (1, 2, 3):
            fig.update_xaxes(range=[0, W], constrain='domain', row=1, col=col)
            ax = '' if col == 1 else str(col)
            fig.update_yaxes(range=[H, 0], scaleanchor=f'x{ax}', scaleratio=1,
                             row=1, col=col)
        d62, d63 = float(np.median(v62)), float(np.median(v63))
        fig.layout.title = (f'{gid}  frame {frame_idx}/{GROUP_SIZE - 1}  |  {stem}<br>'
                            f'<sub>median  v62={d62:.2f} m   v63={d63:.2f} m   '
                            f'(v63-v62)={d63 - d62:+.2f} m   scale x{DEPTH_SCALE:g}</sub>')


def _on_group(change):
    frame_sl.value = 0
    render(change['new'], 0)


def _on_frame(change):
    render(group_dd.value, change['new'])


group_dd.observe(_on_group, names='value')
frame_sl.observe(_on_frame, names='value')

render(group_dd.value, frame_sl.value)
display(widgets.VBox([widgets.HBox([group_dd, play, frame_sl]), fig]))

ImportError: Please install anywidget to use the FigureWidget class

## Optional: v63 − v62 difference map for the current frame

Re-run this cell after moving the slider/dropdown above to render a signed
difference heatmap (`v63 − v62`) for the frame currently selected. Red = v63
predicts *farther*, blue = v63 predicts *nearer*. Hover reads the per-pixel
delta in meters.

In [ ]:
stem_cur = complete[group_dd.value][frame_sl.value]
v62 = load_pred(V62_DIR, stem_cur)
v63 = load_pred(V63_DIR, stem_cur)
diff = v63 - v62
H, W = diff.shape
lim = float(np.percentile(np.abs(diff), 98)) or 1e-6

dfig = go.Figure(go.Heatmap(
    z=diff, zmin=-lim, zmax=lim, colorscale='RdBu_r', zmid=0,
    colorbar=dict(title='v63 - v62 (m)'),
    hovertemplate='x=%{x} y=%{y}<br>Δ=%{z:+.2f} m<extra></extra>',
))
dfig.update_xaxes(range=[0, W], constrain='domain', title='x (px)')
dfig.update_yaxes(range=[H, 0], scaleanchor='x', scaleratio=1, title='y (px)')
dfig.update_layout(
    title=(f'v63 - v62   {stem_cur}<br>'
           f'<sub>mean Δ={diff.mean():+.3f} m   |Δ| p98={lim:.2f} m   '
           f'red = v63 farther, blue = v63 nearer</sub>'),
    height=560, width=680, margin=dict(t=70, l=40, r=40, b=40),
)
dfig.show()